In [ ]:
%pip install -U langchain langchain-core langchain-upstage langchain-pinecone pinecone python-dotenv

In [ ]:
%pip install -U langchain-community


In [ ]:
import os
from typing import List, Tuple, Any

from dotenv import load_dotenv
from pydantic import ConfigDict
from pinecone import Pinecone

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_pinecone import PineconeVectorStore
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


# -----------------------------
# 1) 환경변수 로드
# -----------------------------
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GROUNDLINE_INDEX = os.getenv("GROUNDLINE_INDEX")
BROADCOM_INDEX = os.getenv("BROADCOM_INDEX")
PINECONE_NAMESPACE = os.getenv("PINECONE_NAMESPACE", "default")

if not UPSTAGE_API_KEY:
    raise ValueError("UPSTAGE_API_KEY가 없습니다.")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY가 없습니다.")
if not GROUNDLINE_INDEX or not BROADCOM_INDEX:
    raise ValueError("GROUNDLINE_INDEX / BROADCOM_INDEX가 필요합니다.")


# -----------------------------
# 2) 임베딩 / LLM
# -----------------------------
embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
llm = ChatUpstage()


# -----------------------------
# 3) Pinecone 인덱스 연결
# -----------------------------
pc = Pinecone(api_key=PINECONE_API_KEY)

groundline_index = pc.Index(GROUNDLINE_INDEX)
broadcom_index = pc.Index(BROADCOM_INDEX)

groundline_db = PineconeVectorStore(
    index=groundline_index,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,
)

broadcom_db = PineconeVectorStore(
    index=broadcom_index,
    embedding=embeddings,
    namespace=PINECONE_NAMESPACE,
)


# -----------------------------
# 4) 멀티 인덱스 Retriever
# -----------------------------
class MultiPineconeRetriever(BaseRetriever):
    """
    두 개 이상의 PineconeVectorStore를 동시에 검색한 뒤,
    score 기준으로 정렬해서 상위 문서를 반환하는 커스텀 Retriever
    """

    vectorstores: List[Any]
    k_each: int = 5       # 각 인덱스에서 가져올 개수
    final_k: int = 5      # 최종 반환 개수

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str) -> List[Document]:
        all_docs_with_scores: List[Tuple[Document, float]] = []

        for vs in self.vectorstores:
            # PineconeVectorStore의 similarity_search_with_score 사용
            # score는 vector store 구현에 따라 거리/유사도 의미가 다를 수 있으므로
            # 실제 결과를 보고 정렬 방향을 확인하는 것이 안전함
            docs_with_scores = vs.similarity_search_with_score(query, k=self.k_each)
            all_docs_with_scores.extend(docs_with_scores)

        # 점수 정렬
        # 현재 사용 코드 흐름에 맞춰 reverse=True 유지
        # 만약 결과가 이상하면 reverse=False로 바꿔 테스트하세요.
        all_docs_with_scores = sorted(
            all_docs_with_scores,
            key=lambda x: x[1],
            reverse=True,
        )

        # 최종 상위 문서 추출
        top_docs = []
        for rank, (doc, score) in enumerate(all_docs_with_scores[: self.final_k], start=1):
            # 출처 확인용 metadata 추가
            doc.metadata["retrieval_score"] = score
            doc.metadata["retrieval_rank"] = rank
            top_docs.append(doc)

        return top_docs


multi_retriever = MultiPineconeRetriever(
    vectorstores=[groundline_db, broadcom_db],
    k_each=5,
    final_k=5,
)


# -----------------------------
# 5) QA 프롬프트
# -----------------------------
# qa_prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """
# 너는 통신설비 기술 질의응답 전문가다.
# 반드시 제공된 context만 근거로 답변하라.
# context에 없으면 추측하지 말고 "제공된 문서에서 확인되지 않습니다."라고 답하라.
# 답변은 한국어로, 핵심부터 간결하게 작성하라.

# <context>
# {context}
# </context>
# """.strip(),
#         ),
#         ("human", "{input}"),
#     ]
# )


# # -----------------------------
# # 6) 문서 결합 체인 + Retrieval 체인
# # -----------------------------
# question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
# rag_chain = create_retrieval_chain(multi_retriever, question_answer_chain)

prompt = ChatPromptTemplate.from_template("""
너는 통신설비 기술 질의응답 전문가다.
반드시 제공된 context만 근거로 답변하라.
context에 없으면 추측하지 말고 "제공된 문서에서 확인되지 않습니다."라고 답하라.
답변은 한국어로, 핵심부터 간결하게 작성하라.
<context>
{context}
</context>

질문: {question}
""")

retriever = multi_retriever.as_retriever()

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)
# -----------------------------
# 7) 실행 예시
# -----------------------------
query = "통신설비의 접지저항은 얼마인가?"

result = rag_chain.invoke(query)

print(result)

# query = "통신설비의 접지저항은 얼마입니까?"

# result = rag_chain.invoke({"input": query})

# print("=== 질문 ===")
# print(query)

# print("\n=== 답변 ===")
# print(result["answer"])

# print("\n=== 검색된 문서 ===")
# for i, doc in enumerate(result["context"], start=1):
#     print(f"\n[{i}]")
#     print("score:", doc.metadata.get("retrieval_score"))
#     print("source:", doc.metadata.get("source"))
#     print("index_hint:", doc.metadata.get("index_name"))
#     print(doc.page_content[:300])

In [6]:
print(os.getenv("GROUNDLINE_INDEX"))

groundline-index
